# Lezione 08 — Test di ipotesi II: confrontare più gruppi

**Corso:** Statistica Applicata con Python · **Blocco 3:** Inferenza
**Tempo stimato:** 105 minuti · **Capitolo del manuale:** `manual/08_tests_hipotesis_ii_IT.md`

## Cosa vedrai in questa lezione

- **Il problema dei t-test a coppie** quando i gruppi sono più di due.
- **ANOVA a un fattore** (`f_oneway`): l'idea (varianza tra e dentro i gruppi), la tabella ANOVA a mano e la dimensione dell'effetto **η²**.
- **Confronti post-hoc di Tukey**: un'ANOVA significativa non dice *quali* gruppi differiscono.
- **Presupposti**: normalità dei residui (grafico Q-Q, Shapiro-Wilk), omogeneità delle varianze (Levene) e cosa fare se non valgono (Welch-ANOVA).
- **Test non parametrici**: Mann-Whitney e Kruskal-Wallis, e quando battono l'ANOVA (simulazione di potenza con dati a coda lunga).
- **Leggere e comunicare** un risultato completo (F, p, η², post-hoc) e una tabella per **scegliere il test**.

## Prerequisiti

- Lezione 7 (logica del test, p-value, errori di tipo I e II, t-test di Welch, dimensione dell'effetto) e lezione 6 (errore standard, intervalli di confidenza).
- Librerie: `numpy`, `pandas`, `matplotlib`, `scipy`, `statsmodels` (`pip install numpy pandas matplotlib scipy statsmodels`; sono già installate in Google Colab).

> **Come usare questo notebook:** esegui le celle in ordine. Prima di ogni esperimento scrivi cosa ti aspetti che accada. Tutti i dati (tempi di preparazione degli ordini, tempi di attesa) sono sintetici: un *esempio illustrativo*, non dati reali. I confronti tra categorie e i confronti multipli si vedono nella lezione 9.

---
## Cella 0 — Setup

Importiamo le librerie e fissiamo il **seme**: con lo stesso seme, tutti otteniamo gli stessi numeri.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)          # seme fisso: tutti otteniamo gli stessi risultati
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True
print("Librerie caricate correttamente.")

---
## 1. Il problema: tre magazzini, impiegano lo stesso tempo?

Un'azienda di commercio elettronico ha tre magazzini (A, B e C). Da ciascuno prendiamo 40 ordini e misuriamo il **tempo di preparazione** in minuti (*esempio illustrativo*, dati sintetici). Ci sono differenze reali tra i magazzini o è rumore campionario?

Con due gruppi useresti un t-test di Welch (lezione 7). Con tre, la tentazione è fare tre t-test a coppie; tra poco vedremo perché **non** è una buona idea.

In [ ]:
rng_datos = np.random.default_rng(42)
tiempo_a = rng_datos.normal(32, 6, 40)      # esempio illustrativo (minuti)
tiempo_b = rng_datos.normal(33, 6, 40)
tiempo_c = rng_datos.normal(38, 6, 40)
grupos = [tiempo_a, tiempo_b, tiempo_c]
nombres = ["A", "B", "C"]

resumen = pd.DataFrame({
    "n": [len(g) for g in grupos],
    "media (min)": [g.mean() for g in grupos],
    "dev. standard (min)": [g.std(ddof=1) for g in grupos]},
    index=["magazzino " + nom for nom in nombres])
display(resumen.round(2))

plt.boxplot(grupos, tick_labels=nombres)
plt.title("Tempo di preparazione per magazzino")
plt.ylabel("minuti")
plt.show()

---
## 2. L'idea dell'ANOVA: varianza tra i gruppi contro varianza dentro i gruppi

L'ANOVA a un fattore verifica **H0: tutte le medie della popolazione sono uguali** contro H1: **almeno una è diversa**. Confronta due quantità:

- la **variabilità tra i gruppi** (quanto le medie dei gruppi si allontanano dalla media globale);
- la **variabilità dentro i gruppi** (quanto i dati variano attorno alla media del *proprio* gruppo: il rumore).

La statistica **F** è il rapporto tra le due, ciascuna divisa per i propri gradi di libertà. Se H0 è vera, F è intorno a 1; se le medie differiscono davvero, F cresce. Prima la calcoliamo **a mano** con un esempio minuscolo: 3 gruppi da 4 ordini.

In [ ]:
peq = {"A": np.array([12, 14, 13, 15]),
       "B": np.array([16, 18, 17, 15]),
       "C": np.array([20, 19, 22, 19])}          # esempio illustrativo (minuti)

valores = np.concatenate(list(peq.values()))
media_global = valores.mean()
ss_entre = sum(len(v) * (v.mean() - media_global) ** 2 for v in peq.values())     # somma dei quadrati tra i gruppi
ss_dentro = sum(((v - v.mean()) ** 2).sum() for v in peq.values())                # somma dei quadrati dentro i gruppi
k, N = len(peq), len(valores)
gl_entre, gl_dentro = k - 1, N - k
ms_entre, ms_dentro = ss_entre / gl_entre, ss_dentro / gl_dentro                   # quadrati medi
F = ms_entre / ms_dentro
p_F = stats.f.sf(F, gl_entre, gl_dentro)
eta2 = ss_entre / (ss_entre + ss_dentro)

tabla_anova = pd.DataFrame({
    "somma dei quadrati": [ss_entre, ss_dentro, ss_entre + ss_dentro],
    "gl": [gl_entre, gl_dentro, N - 1],
    "quadrato medio": [ms_entre, ms_dentro, np.nan]},
    index=["tra i gruppi", "dentro i gruppi", "totale"])
display(tabla_anova.round(3))
print(f"media globale = {media_global:.3f}   F = {F:.3f}   p = {p_F:.5f}   η² = {eta2:.3f}")
print(f"valore critico di F (α = 0,05) = {stats.f.ppf(0.95, gl_entre, gl_dentro):.3f}")

ref = stats.f_oneway(*peq.values())
print(f"scipy: F = {ref.statistic:.3f}   p = {ref.pvalue:.5f}")
assert np.isclose(F, ref.statistic) and np.isclose(p_F, ref.pvalue)

### Esperimento guidato: il rumore dentro i gruppi decide

Manteniamo **esattamente le stesse medie di gruppo** (13,5 · 16,5 · 20) ma moltiplichiamo per 1, 2, 4 e 8 la dispersione dei dati attorno a ciascuna media. Cosa succede a F e al p-value?

In [ ]:
filas = []
for factor in [1, 2, 4, 8]:
    ruidosos = [v.mean() + factor * (v - v.mean()) for v in peq.values()]
    r = stats.f_oneway(*ruidosos)
    filas.append({"fattore di rumore": factor,
                  "dev. standard media dentro": np.mean([g.std(ddof=1) for g in ruidosos]),
                  "F": r.statistic, "p-value": r.pvalue})
display(pd.DataFrame(filas).round(4).set_index("fattore di rumore"))

**Cosa osservi?**

- Le medie di gruppo non sono cambiate, ma F scende e il p-value sale al crescere della dispersione interna. Con fattore 1, F = 23,81; con fattore 8, F ≈ 0,37.
- Una differenza tra medie è "grande" solo **rispetto al rumore dentro i gruppi**. È la stessa idea della lezione 7 (differenza / errore standard), generalizzata a più di due gruppi.

*(La tua risposta qui.)*

---
## 3. ANOVA con `scipy` e dimensione dell'effetto η²

Applichiamo `stats.f_oneway` ai tre magazzini. Il p-value dice se le medie sono compatibili con H0; **non** dice quanto pesa il fattore. Per questo usiamo **η² (eta quadro)** = somma dei quadrati tra i gruppi / somma dei quadrati totale: la proporzione della variabilità totale spiegata dall'appartenenza al gruppo.

In [ ]:
def eta_cuadrado(*gs):
    """η² = SS tra i gruppi / SS totale."""
    todo = np.concatenate(gs)
    ss_total = ((todo - todo.mean()) ** 2).sum()
    ss_e = sum(len(g) * (g.mean() - todo.mean()) ** 2 for g in gs)
    return ss_e / ss_total

anova = stats.f_oneway(*grupos)
print(f"F = {anova.statistic:.3f}   p = {anova.pvalue:.6f}   η² = {eta_cuadrado(*grupos):.3f}")

**Cosa osservi?**

- Il p-value è molto piccolo: rifiutiamo H0 (tutte le medie uguali).
- η² ≈ 0,14: il magazzino spiega circa il 14 % della variabilità dei tempi; l'altro 86 % è variazione tra ordini dentro ciascun magazzino. (Come orientamento abituale, 0,01 piccolo, 0,06 medio, 0,14 grande: sono convenzioni, non verità.)
- Sai già **quali** magazzini sono diversi? No: l'ANOVA dice solo che **almeno uno** lo è.

*(La tua risposta qui.)*

---
## 4. Quali gruppi differiscono? Confronti post-hoc di Tukey

Il **test HSD di Tukey** confronta tutte le coppie di medie controllando la probabilità di qualche falso positivo **tra i confronti dell'insieme** (calibrato sull'intervallo delle medie). Dà un p-value corretto e un intervallo di confidenza per ogni differenza. Si usa **dopo** un'ANOVA significativa, non al suo posto.

In [ ]:
tk = stats.tukey_hsd(*grupos)
ic_tk = tk.confidence_interval(confidence_level=0.95)
filas = []
for i in range(3):
    for j in range(i + 1, 3):
        filas.append({"confronto": f"{nombres[i]} − {nombres[j]}",
                      "differenza tra le medie": grupos[i].mean() - grupos[j].mean(),
                      "IC 95 % inf.": ic_tk.low[i, j], "IC 95 % sup.": ic_tk.high[i, j],
                      "p corretto (Tukey)": tk.pvalue[i, j]})
display(pd.DataFrame(filas).set_index("confronto").round(4))

# lo stesso risultato con statsmodels
from statsmodels.stats.multicomp import pairwise_tukeyhsd
etiquetas = np.repeat(nombres, 40)
print(pairwise_tukeyhsd(np.concatenate(grupos), etiquetas))

**Cosa osservi?**

- A e B **non** si distinguono (p corretto ≈ 0,70 e l'IC include 0); C è più lento di A e di B (p corretti ≈ 0,000 e ≈ 0,003).
- Per questo l'ANOVA significativa **non** significa "tutti differiscono": qui differisce uno solo, il magazzino C.
- `scipy` e `statsmodels` danno gli stessi p-value (il segno delle differenze dipende dall'ordine della coppia).
- Se invece avessimo fatto tre t-test senza correzione, ciascuno con α = 0,05, la probabilità di qualche falso positivo con H0 vera sarebbe 1 − 0,95³ ≈ 14 %, non 5 %.

*(La tua risposta qui.)*

---
## 5. Presupposti dell'ANOVA e cosa fare se non valgono

L'ANOVA classica presuppone: (1) osservazioni **indipendenti**; (2) **residui** (ogni dato meno la media del suo gruppo) approssimativamente **normali**; (3) **varianze** simili tra i gruppi. L'indipendenza dipende dal disegno; le altre due si guardano con grafici e test:

- **Grafico Q-Q dei residui**: punti vicini alla retta = normalità ragionevole.
- **Shapiro-Wilk** sui residui: complemento con dei limiti (poco potente con n piccolo, ipersensibile con n grande).
- **Levene**: H0 = le varianze sono uguali.

Se non valgono: **Welch-ANOVA** (non richiede varianze uguali) o il test non parametrico di **Kruskal-Wallis** (sezione 6).

In [ ]:
def residuos_de(*gs):
    return np.concatenate([g - g.mean() for g in gs])

# Caso 1: i tre magazzini
res_alm = residuos_de(*grupos)
print(f"Magazzini: Shapiro-Wilk (residui) p = {stats.shapiro(res_alm).pvalue:.3f}   Levene p = {stats.levene(*grupos).pvalue:.3f}")

# Caso 2: tempi di attesa con coda lunga (esempio illustrativo)
rng_cola = np.random.default_rng(3)
cola_1 = rng_cola.lognormal(3.0, 0.7, 20)
cola_2 = rng_cola.lognormal(3.1, 0.7, 20)
cola_3 = rng_cola.lognormal(3.7, 0.7, 20)
colas = [cola_1, cola_2, cola_3]
res_cola = residuos_de(*colas)
print(f"Code lunghe: Shapiro-Wilk (residui) p = {stats.shapiro(res_cola).pvalue:.1e}   Levene p = {stats.levene(*colas).pvalue:.3f}   dev. standard = {[round(float(g.std(ddof=1)), 1) for g in colas]}")

fig, ejes = plt.subplots(1, 2, figsize=(10, 4))
stats.probplot(res_alm, dist="norm", plot=ejes[0]); ejes[0].set_title("Q-Q dei residui: magazzini")
stats.probplot(res_cola, dist="norm", plot=ejes[1]); ejes[1].set_title("Q-Q dei residui: code lunghe")
plt.tight_layout()
plt.show()

Applichiamo al secondo caso l'ANOVA classica e la **Welch-ANOVA** (`statsmodels.stats.oneway.anova_oneway` con `use_var="unequal"`), che non richiede varianze uguali:

In [ ]:
from statsmodels.stats.oneway import anova_oneway

clasico = stats.f_oneway(*colas)
welch = anova_oneway(colas, use_var="unequal", welch_correction=True)
print(f"ANOVA classica: F = {clasico.statistic:.3f}   p = {clasico.pvalue:.4f}")
print(f"Welch-ANOVA:  F = {welch.statistic:.3f}   p = {welch.pvalue:.4f}")

**Cosa osservi?**

- Nei magazzini il Q-Q segue la retta e né Shapiro né Levene rifiutano: i presupposti sono ragionevoli.
- Nei tempi con coda lunga il Q-Q si curva e Shapiro rifiuta con chiarezza. Le deviazioni standard vanno da ≈ 20 a ≈ 45, ma Levene non rifiuta (p ≈ 0,89): con n = 20 per gruppo e dati asimmetrici ha poca potenza, quindi un p alto in Levene non garantisce varianze uguali.
- L'ANOVA classica e la Welch-ANOVA danno conclusioni diverse (p ≈ 0,10 contro p ≈ 0,03). Quando i presupposti non valgono, la scelta del test **cambia la conclusione**: va decisa con criterio e, se possibile, prima di vedere il p-value.

*(La tua risposta qui.)*

---
## 6. Test non parametrici: Mann-Whitney e Kruskal-Wallis

Lavorano sui **ranghi** (posizione di ogni dato ordinando tutti insieme) e non presuppongono normalità. **Mann-Whitney U** confronta 2 gruppi indipendenti; **Kruskal-Wallis** confronta ≥ 3. Ciò che verificano davvero: se un dato di un gruppo tende a essere **maggiore** di uno dell'altro (**dominanza stocastica**, P(X > Y) ≠ 0,5), **non** "le mediane" in generale. Si preferiscono con code pesanti, dati ordinali o n piccolo; se i dati sono normali, perdono un po' di potenza.

In [ ]:
rng_np = np.random.default_rng(7)
espera_x = rng_np.lognormal(3.0, 0.8, 25)          # esempio illustrativo: tempi di attesa con coda lunga
espera_y = rng_np.lognormal(3.5, 0.8, 25)
welch_x = stats.ttest_ind(espera_x, espera_y, equal_var=False)
mw = stats.mannwhitneyu(espera_x, espera_y, alternative="two-sided")
print(f"mediane: {np.median(espera_x):.1f} e {np.median(espera_y):.1f}")
print(f"t di Welch: p = {welch_x.pvalue:.4f}")
print(f"Mann-Whitney:  U = {mw.statistic:.0f}   p = {mw.pvalue:.4f}   P(X > Y) stimata = {mw.statistic / (25 * 25):.3f}")

kw = stats.kruskal(*colas)
print(f"Kruskal-Wallis (3 gruppi con code lunghe): H = {kw.statistic:.2f}   p = {kw.pvalue:.5f}   (ANOVA classica p = {stats.f_oneway(*colas).pvalue:.3f})")

### Mann-Whitney non è un test sulle mediane

Generiamo due gruppi da 1 000 dati con **la stessa mediana** (≈ 10) ma forme diverse: uno simmetrico e uno con coda lunga a destra.

In [ ]:
rs = np.random.default_rng(11)
sim_x = rs.normal(10, 1, 1000)
sim_y = 10 + (rs.lognormal(0, 1.0, 1000) - 1.0)
mw2 = stats.mannwhitneyu(sim_x, sim_y)
print(f"mediane: {np.median(sim_x):.2f} e {np.median(sim_y):.2f}   medie: {sim_x.mean():.2f} e {sim_y.mean():.2f}")
print(f"Mann-Whitney: p = {mw2.pvalue:.6f}   P(X > Y) = {mw2.statistic / 1000 ** 2:.3f}")
print(f"test della mediana di Mood: p = {stats.median_test(sim_x, sim_y)[1]:.3f}")

plt.hist(sim_x, bins=40, alpha=0.6, label="X")
plt.hist(sim_y, bins=40, alpha=0.6, label="Y")
plt.title("Stessa mediana, forme diverse"); plt.legend()
plt.show()

**Cosa osservi?**

- Mann-Whitney risulta molto significativo (p ≈ 9 · 10⁻⁶) anche se le mediane sono quasi identiche: rileva che P(X > Y) ≈ 0,44, non che le mediane differiscano. Il test della mediana non trova nulla (p ≈ 0,45).
- Interpretalo come "un dato di Y tende a essere maggiore di uno di X" (probabilità ≈ 0,56), non come "le mediane sono diverse". Si può leggere come differenza di mediane solo se supponi che le due distribuzioni abbiano la **stessa forma** (traslazione).
- Nei tempi con coda lunga, Mann-Whitney e Kruskal-Wallis danno p più piccoli dei loro equivalenti parametrici: con code pesanti hanno di solito **più** potenza.

*(La tua risposta qui.)*

### Esperimento guidato: la potenza del t di Welch contro Mann-Whitney

Ripetiamo 2 000 volte un confronto con n = 20 per gruppo e un effetto reale, con tre forme di dati, e contiamo con che frequenza rifiuta ciascun test (potenza).

In [ ]:
def potencia_comparada(generador, n=20, reps=2000, semilla=1):
    r = np.random.default_rng(semilla)
    rechaza_t = rechaza_mw = 0
    for _ in range(reps):
        a, b = generador(r, n)
        rechaza_t += stats.ttest_ind(a, b, equal_var=False).pvalue < 0.05
        rechaza_mw += stats.mannwhitneyu(a, b).pvalue < 0.05
    return rechaza_t / reps, rechaza_mw / reps

escenarios = {
    "Normale": lambda r, n: (r.normal(0, 1, n), r.normal(0.8, 1, n)),
    "Code pesanti (t con 3 gdl)": lambda r, n: (r.standard_t(3, n), r.standard_t(3, n) + 1.2),
    "Asimmetrica (lognormale)": lambda r, n: (r.lognormal(0, 1, n), r.lognormal(0.8, 1, n)),
}
filas = []
for nombre, gen in escenarios.items():
    pt, pmw = potencia_comparada(gen)
    filas.append({"scenario": nombre, "potenza t di Welch": pt, "potenza Mann-Whitney": pmw})
display(pd.DataFrame(filas).set_index("scenario").round(3))

**Cosa osservi?**

- Con dati normali le due potenze sono quasi uguali (≈ 0,70 e ≈ 0,69): Mann-Whitney perde pochissimo.
- Con code pesanti o asimmetria, Mann-Whitney rileva l'effetto molto più spesso (≈ 0,81 contro ≈ 0,66; ≈ 0,69 contro ≈ 0,48).
- "Non parametrico" non significa "peggiore": quando i presupposti del t-test sono dubbi, è spesso migliore.

*(La tua risposta qui.)*

---
## 7. ANOVA o Kruskal-Wallis? Una simulazione con dati a coda lunga

Con **tre gruppi** da 15 dati confrontiamo la frequenza di rifiuto dei due test in tre forme di dati. Per ogni forma simuliamo 2 000 "studi" in due situazioni: **senza effetto** (H0 vera: la frequenza di rifiuto è il tasso di falsi positivi, che dovrebbe stare intorno a 0,05) e **con un effetto reale** (la frequenza di rifiuto è la potenza).

In [ ]:
def comparar_anova_kw(generador, n=15, reps=2000, semilla=5):
    """Frequenza di rifiuto (p < 0,05) di ANOVA e di Kruskal-Wallis."""
    r = np.random.default_rng(semilla)
    rechaza_f = rechaza_kw = 0
    for _ in range(reps):
        gs = generador(r, n)
        rechaza_f += stats.f_oneway(*gs).pvalue < 0.05
        rechaza_kw += stats.kruskal(*gs).pvalue < 0.05
    return rechaza_f / reps, rechaza_kw / reps

# (senza effetto, con effetto) per ogni forma di dati
escenarios3 = {
    "Normale": (lambda r, n: [r.normal(0, 1, n) for _ in range(3)],
                         lambda r, n: [r.normal(m, 1, n) for m in (0, 0.5, 1.0)]),
    "Code pesanti (t con 3 gdl)": (lambda r, n: [r.standard_t(3, n) for _ in range(3)],
                                                             lambda r, n: [r.standard_t(3, n) + m for m in (0, 0.8, 1.6)]),
    "Asimmetrica (lognormale)": (lambda r, n: [r.lognormal(0, 1, n) for _ in range(3)],
                                                       lambda r, n: [r.lognormal(m, 1, n) for m in (0, 0.5, 1.0)]),
}
filas = []
for nombre, (sin_efecto, con_efecto) in escenarios3.items():
    fp_f, fp_kw = comparar_anova_kw(sin_efecto)
    pot_f, pot_kw = comparar_anova_kw(con_efecto)
    filas.append({"dati": nombre,
                  "falsi positivi ANOVA": fp_f, "falsi positivi Kruskal-Wallis": fp_kw,
                  "potenza ANOVA": pot_f, "potenza Kruskal-Wallis": pot_kw})
tabla_pot = pd.DataFrame(filas).set_index("dati")
display(tabla_pot.round(3))

tabla_pot.iloc[:, 2:].plot.bar(rot=0, figsize=(9, 4))
plt.title("Potenza: ANOVA contro Kruskal-Wallis (3 gruppi, n = 15)")
plt.ylabel("potenza")
plt.show()

**Cosa osservi?**

- Senza effetto, i due test rifiutano tra il 3 % e il 5 % delle volte: entrambi controllano l'errore di tipo I anche con dati asimmetrici (con n = 15 l'approssimazione dell'ANOVA è ragionevole).
- Con dati normali le potenze sono simili (≈ 0,65 e ≈ 0,61): Kruskal-Wallis perde poco.
- Con code pesanti o asimmetria, Kruskal-Wallis **vince** (≈ 0,81 contro ≈ 0,68 con t a 3 gdl; ≈ 0,61 contro ≈ 0,42 con lognormale). Con un'ANOVA si perderebbero effetti reali.
- Attenzione: questo vale per queste forme e questi effetti. Scegli il test **prima** di guardare i p-value; se provi entrambi e tieni il più piccolo stai facendo due test e tenendo il migliore.

*(La tua risposta qui.)*

---
## 8. Leggere e comunicare un risultato completo

Un buon rapporto di confronto tra gruppi riunisce **quattro elementi**: il risultato globale (F con i suoi gradi di libertà e p-value), la dimensione dell'effetto (η²), le medie (e deviazioni) di ciascun gruppo e il post-hoc con la differenza, il suo intervallo di confidenza e il p corretto. Generiamo il testo a partire dai risultati dei magazzini (sezione 3-4) e dal Kruskal-Wallis delle code lunghe (sezione 5).

In [ ]:
k = len(grupos)
N = sum(len(g) for g in grupos)
anova = stats.f_oneway(*grupos)
lineas = [f"ANOVA: F({k - 1}, {N - k}) = {anova.statistic:.2f}, p = {anova.pvalue:.4f}, η² = {eta_cuadrado(*grupos):.2f}.",
          "Medie (min): " + ", ".join(f"{nombres[i]} = {grupos[i].mean():.1f} ± {grupos[i].std(ddof=1):.1f}" for i in range(k)) + "."]
for i in range(k):
    for j in range(i + 1, k):
        dif = grupos[i].mean() - grupos[j].mean()
        veredicto = "differiscono" if tk.pvalue[i, j] < 0.05 else "non si distinguono"
        lineas.append(f"Tukey {nombres[i]} − {nombres[j]}: {dif:+.2f} min, IC 95 % ({ic_tk.low[i, j]:.2f}; {ic_tk.high[i, j]:.2f}), p corretto = {tk.pvalue[i, j]:.4f} → {veredicto}.")
print("\n".join(lineas))

# Versione non parametrica con le code lunghe
kw = stats.kruskal(*colas)
print(f"\nKruskal-Wallis: H({len(colas) - 1}) = {kw.statistic:.2f}, p = {kw.pvalue:.4f}. Mediane: " + ", ".join(f"{np.median(g):.1f}" for g in colas) + ".")

**Cosa osservi?**

- Dal risultato globale (F, p) passiamo alle coppie: differiscono solo le coppie con C; A − B ha un IC che include 0.
- Il paragrafo di conclusione sarebbe: *"Il tempo di preparazione differisce tra i magazzini (ANOVA, F(2, 117) = 9,78; p = 0,0001; η² = 0,14, effetto grande). Tukey indica che C è più lento di A (≈ 4,4 min) e di B (≈ 3,5 min); A e B non si distinguono."* Si dice **cosa** differisce, **quanto** e con quale incertezza.
- Un rapporto che dice solo "p < 0,05, ci sono differenze significative" non permette di decidere nulla.

*(La tua risposta qui.)*

---
## 9. Come scegliere il test per confrontare gruppi numerici

| Situazione | Presupposti ragionevoli | Presupposti dubbi (code pesanti, valori estremi, ordinali, n piccolo) |
|---|---|---|
| 2 gruppi indipendenti | t di Welch (lezione 7) | Mann-Whitney (`stats.mannwhitneyu`) |
| 2 misure dello stesso soggetto (appaiate) | t appaiato (lezione 7) | Wilcoxon dei ranghi con segno (`stats.wilcoxon`) |
| ≥ 3 gruppi indipendenti | ANOVA + Tukey; Welch-ANOVA se le varianze differiscono | Kruskal-Wallis (+ Dunn come post-hoc) |
| Variabili categoriali (conteggi) | lezione 9: chi-quadro, Fisher | — |
| Molti test insieme | lezione 9: Bonferroni, Benjamini-Hochberg | — |

Tukey corregge già per il numero di coppie confrontate: **non** serve aggiungere Bonferroni sopra. Se fai più Mann-Whitney a coppie (perché in `scipy` non c'è Dunn), serve invece una correzione della lezione 9.

---
## Riepilogo della lezione

- **Molti t-test a coppie** gonfiano il rischio di falsi positivi (1 − 0,95³ ≈ 14 % con 3 gruppi). Un unico test globale lo evita.
- **ANOVA a un fattore** (`f_oneway`): F = varianza tra i gruppi / varianza dentro; H0: tutte le medie uguali. Riporta anche **η²**. Un'ANOVA significativa dice che **almeno un** gruppo differisce, non quali: usa **Tukey** (`stats.tukey_hsd`), che controlla già l'errore dell'insieme delle coppie.
- **Presupposti**: indipendenza, residui approssimativamente normali (Q-Q; Shapiro con i suoi limiti) e varianze simili (Levene, che può avere poca potenza). Se non valgono: Welch-ANOVA o Kruskal-Wallis.
- **Mann-Whitney e Kruskal-Wallis** confrontano ranghi (dominanza stocastica), non "le mediane" senza condizioni; guadagnano con code pesanti e perdono poco con dati normali.
- Un buon rapporto riunisce **F(gdl), p, η², medie e post-hoc con IC**; il test si sceglie **prima** di vedere i p-value.

**Prossimo passo:** il notebook di pratica della lezione 8 e poi la lezione 9 (categorie e confronti multipli).